<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## The `GraphCompiler` Class 🛠️

This class traverses the `LogicalPlan` and builds a MAX Graph. It maps our `Expr` nodes to pure `max.graph.ops`.

In [0]:
#| echo: false
#| output: asis
show_doc(GraphCompiler)

---

### GraphCompiler

```python

def GraphCompiler(
    device:str='cpu'
):


```

*Compiles a LogicalPlan into a MAX Graph.*

Key design:
- All Filter nodes are evaluated eagerly in PyArrow *before* the graph is built
  (_strip_filters). This correctly removes rows and makes count() trivially correct.
- Supports: Scan, Project, Filter (pre-applied), Aggregate, Sort, Limit, Distinct.
- Sort/Limit/Distinct are stored as post-ops and applied after graph execution.
- device: "cpu" | "gpu" | "auto"
    "cpu"  -- always use CPU (default)
    "gpu"  -- require a MAX-compatible GPU; raises RuntimeError if unavailable or
             if the GPU is not supported by the installed MAX version.
    "auto" -- use GPU if MAX-compatible, else fall back to CPU silently

## Tests 🧪

Let's verify that our compiler can translate a simple plan and execute it.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit
from mxframe.lazy_frame import LazyFrame, Scan

# ── Test 1: Simple projection ──
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))
result = GraphCompiler().compile_and_run(lf.select(col('a') + lit(10)).plan)
assert result.column(0).to_pylist() == [11, 12, 13], f'Projection failed: {result}'
print('✅ Test 1 passed: projection')

# ── Test 2: Global sum aggregation ──
lf2 = LazyFrame(Scan(pa.table({'x': [1.0, 2.0, 3.0, 4.0]})))
result2 = GraphCompiler().compile_and_run(
    lf2.groupby().agg(col('x').sum().alias('total')).plan
)
assert result2.column('total').to_pylist()[0] == 10.0, f'Sum failed: {result2}'
print('✅ Test 2 passed: global sum aggregation')

# ── Test 3: Filter REMOVES rows (not zeros them) ──
table3 = pa.table({'a': [1, 2, 3, 4, 5], 'b': [10.0, 20.0, 30.0, 40.0, 50.0]})
lf3 = LazyFrame(Scan(table3))
result3 = GraphCompiler().compile_and_run(lf3.filter(col('a') > lit(2)).plan)
assert result3.num_rows == 3, f'Filter should produce 3 rows, got {result3.num_rows}'
assert result3.column('a').to_pylist() == [3, 4, 5], f'Wrong rows: {result3.column("a").to_pylist()}'
print('✅ Test 3 passed: filter removes rows (not masks them)')

# ── Test 4: mean() is correct after filter ──
# Old bug: mask-multiply gave mean([0,0,30,40,50])/5 = 24.0
# Correct:  mean([30.0, 40.0, 50.0]) = 40.0
result4 = GraphCompiler().compile_and_run(
    lf3.filter(col('a') > lit(2)).groupby().agg(col('b').mean().alias('avg_b')).plan
)
avg_b = result4.column('avg_b').to_pylist()[0]
assert abs(avg_b - 40.0) < 1e-6, f'Mean after filter should be 40.0, got {avg_b} (old bug was 24.0)'
print('✅ Test 4 passed: mean() correct after filter (old bug was 24.0)')

# ── Test 5: count() global ──
table5 = pa.table({'v': [10, 20, 30, 40, 50]})
lf5 = LazyFrame(Scan(table5))
result5 = GraphCompiler().compile_and_run(
    lf5.groupby().agg(col('v').count().alias('n')).plan
)
assert result5.column('n').to_pylist()[0] == 5, f'Count should be 5, got {result5.column("n").to_pylist()[0]}'
print('✅ Test 5 passed: count() global')

# ── Test 6: count() after filter ──
result6 = GraphCompiler().compile_and_run(
    lf5.filter(col('v') > lit(25)).groupby().agg(col('v').count().alias('n')).plan
)
assert result6.column('n').to_pylist()[0] == 3, f'Count after filter should be 3, got {result6.column("n").to_pylist()[0]}'
print('✅ Test 6 passed: count() after filter')

print('\nAll GraphCompiler tests passed! ✅')

✅ Test 1 passed: projection
✅ Test 2 passed: global sum aggregation
✅ Test 3 passed: filter removes rows (not masks them)
✅ Test 4 passed: mean() correct after filter (old bug was 24.0)
✅ Test 5 passed: count() global
✅ Test 6 passed: count() after filter

All GraphCompiler tests passed! ✅
